# 17 Is the claim still true? Small checkers that tell a stale rule from a current one

In the RAG workshop, an off-the-shelf **cross-encoder** (a model that reads a question and a passage together and returns a relevance score) rated the superseded permission 0.8758 and the current prohibition 0.0142 for the same question, so no threshold on that score keeps the right sentence. Relevance measures topic overlap; whether a sentence is *still true* is an entailment question. An **entailment checker** takes evidence and a claim and returns the probability that the evidence supports the claim, in one forward pass on a CPU.

This notebook runs four small checkers on pairs where the claim is the same and only the evidence changed: **VitaminC** (Schuster et al., 2021) flip pairs, taken from Wikipedia revisions, where a claim is supported by the evidence before an edit and refuted by it after. It then reports what each checker gets right with intervals, what a calibrated abstention gate (split conformal prediction) does with their scores, and repeats the test on six Meridian policy pairs.

The four checkers: `ms-marco-MiniLM-L-6-v2` (cross-encoder relevance, 22M parameters), `nli-deberta-v3-base` (a natural-language-inference model, 184M), **HHEM** (Vectara's hallucination checker built on Flan-T5-base) and **MiniCheck-RoBERTa-Large** (355M, trained on synthetic grounding data; Tang, Laban and Durrett, EMNLP 2024).

**Goal.** Fix the settings, download the VitaminC test split, and build flip pairs.

In [1]:
import json, math, random, time, collections, urllib.request
from pathlib import Path
import numpy as np, torch
torch.set_num_threads(3)

N_PAIRS = 50          # 50 pairs = 100 examples, about 6 minutes on a 4-core CPU; the recorded run used 300
SEED = 0
CACHE = Path("data/cache/vitaminc"); CACHE.mkdir(parents=True, exist_ok=True)
RECORD = Path("data/results/17_grounding_checkers")      # scores from the 300-pair run, kept in the repository
URL = "https://huggingface.co/datasets/tals/vitaminc/resolve/main/test.jsonl"   # CC BY-SA 3.0; stored in the git-ignored cache only
f = CACHE / "test.jsonl"
if not f.exists():
    urllib.request.urlretrieve(URL, f)
rows = [json.loads(l) for l in open(f)]
print(len(rows), "rows;", collections.Counter(r["label"] for r in rows))

55197 rows; Counter({'SUPPORTS': 27664, 'REFUTES': 20265, 'NOT ENOUGH INFO': 7268})


In [2]:
# A flip pair: one claim, with evidence that SUPPORTS it and revised evidence that REFUTES it.
by_case = collections.defaultdict(list)
for r in rows:
    if r["revision_type"] == "real":
        by_case[r["case_id"]].append(r)
pairs = []
for cid, v in by_case.items():
    by = {}
    for r in v:
        if r["label"] in ("SUPPORTS", "REFUTES"):
            by.setdefault((r["claim"], r["label"]), r)
    for c in sorted({c for c, _ in by}):      # sorted: set order changes between runs, which would change the pairs
        if (c, "SUPPORTS") in by and (c, "REFUTES") in by:
            pairs.append((cid, by[(c, "SUPPORTS")], by[(c, "REFUTES")])); break
random.Random(SEED).shuffle(pairs)
print(len(pairs), "flip pairs available; using", N_PAIRS)
cid, s, r = pairs[0]
print("claim:   ", s["claim"]); print("before:  ", s["evidence"][:160]); print("after:   ", r["evidence"][:160])

4955 flip pairs available; using 50
claim:    Chenab is a tributary of the Indus River .
before:   In the plains , its left bank tributary is the Chenab which itself has four major tributaries , namely , the Jhelum , the Ravi , the Beas , and the Sutlej .
after:    In the plains , its left bank tributary is the Sutlej which itself has four major tributaries , namely , the Chenab , Jhelum , the Ravi , the Beas , and the Sut


**Goal.** Load the four checkers behind one interface, `score(claim, evidence)`, where a higher number means "the evidence supports the claim". HHEM publishes custom model code; this notebook does not run it. It rebuilds the same network from the standard `T5ForTokenClassification` class and loads the published weights from a `safetensors` file (a format that holds numbers only, no executable code). MiniCheck-RoBERTa-Large ships a pickle file that the pinned PyTorch refuses to load, so its weights come from the `safetensors` conversion pull request on the model page (`refs/pr/1`, made by the Hugging Face conversion bot).

In [3]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, T5ForTokenClassification
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

def load_ce():
    from sentence_transformers import CrossEncoder
    m = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    return lambda claim, ev: float(m.predict([(claim, ev)])[0])          # a raw relevance logit, not a probability

def load_nli():
    M = "cross-encoder/nli-deberta-v3-base"
    tok = AutoTokenizer.from_pretrained(M); m = AutoModelForSequenceClassification.from_pretrained(M).eval()
    ent = [k for k, v in m.config.id2label.items() if v.lower() == "entailment"][0]
    def f(claim, ev):
        with torch.no_grad():
            return float(m(**tok(ev, claim, return_tensors="pt", truncation=True, max_length=512)).logits.softmax(-1)[0][ent])
    return f

def load_hhem():
    cfg = AutoConfig.from_pretrained("google/flan-t5-base"); cfg.num_labels = 2
    m = T5ForTokenClassification(cfg).eval()
    sd = {k[3:] if k.startswith("t5.") else k: v for k, v in
          load_file(hf_hub_download("vectara/hallucination_evaluation_model", "model.safetensors")).items()}
    r = m.load_state_dict(sd, strict=False)
    print("HHEM missing keys:", r.missing_keys, "| unexpected:", r.unexpected_keys)   # one tied embedding; nothing unexpected
    tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
    T = "<pad> Determine if the hypothesis is true given the premise?\n\nPremise: {t1}\n\nHypothesis: {t2}"
    def f(claim, ev):
        with torch.no_grad():
            return float(m(**tok([T.format(t1=ev, t2=claim)], return_tensors="pt", padding=True, truncation=True, max_length=512)).logits[:, 0, :].softmax(-1)[0][1])
    return f

def load_minicheck():
    M = "lytang/MiniCheck-RoBERTa-Large"
    cfg = AutoConfig.from_pretrained(M); tok = AutoTokenizer.from_pretrained(M)
    m = AutoModelForSequenceClassification.from_config(cfg).eval()
    r = m.load_state_dict(load_file(hf_hub_download(M, "model.safetensors", revision="refs/pr/1")), strict=False)
    print("MiniCheck missing keys:", len(r.missing_keys), "| unexpected:", len(r.unexpected_keys))
    def f(claim, ev):
        with torch.no_grad():
            return float(m(**tok(ev, claim, return_tensors="pt", truncation=True, max_length=512)).logits.softmax(-1)[0][1])
    return f

LOADERS = {"ce": load_ce, "nli": load_nli, "hhem": load_hhem, "minicheck": load_minicheck}
scorers = {name: fn() for name, fn in LOADERS.items()}

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HHEM missing keys: ['transformer.encoder.embed_tokens.weight'] | unexpected: []


MiniCheck missing keys: 0 | unexpected: 0


**Goal.** Score every example, one printed line per ten examples, and keep each score in a file. The run is the slow cell; the recorded 300-pair scores below let you skip it.

In [4]:
examples = []
for i, (cid, s, r) in enumerate(pairs[:N_PAIRS]):
    examples += [(i, "S", s["claim"], s["evidence"], 1), (i, "R", r["claim"], r["evidence"], 0)]
OUT = CACHE / f"scores_{N_PAIRS}.jsonl"
fresh = []
with open(OUT, "w") as fh:
    for name, score in scorers.items():
        t0 = time.time()
        for k, (i, side, claim, ev, y) in enumerate(examples, 1):
            t = time.time(); v = score(claim, ev); ms = (time.time() - t) * 1000
            row = dict(model=name, pair=i, side=side, y=y, score=v, ms=round(ms, 1)); fresh.append(row); fh.write(json.dumps(row) + "\n")
            if k % 20 == 0: print(f"{name}: {k}/{len(examples)} last score {v:.3f} ({ms:.0f} ms)", flush=True)
        print(f"{name}: {len(examples)} examples in {time.time() - t0:.0f} s", flush=True)

ce: 20/100 last score 7.359 (111 ms)


ce: 40/100 last score 4.170 (97 ms)


ce: 60/100 last score 4.216 (70 ms)


ce: 80/100 last score -0.324 (68 ms)


ce: 100/100 last score 4.958 (191 ms)


ce: 100 examples in 10 s


... (14 outputs left out)

minicheck: 60/100 last score 0.863 (5486 ms)


minicheck: 80/100 last score 0.121 (1851 ms)


minicheck: 100/100 last score 0.027 (5643 ms)


minicheck: 100 examples in 296 s


**Goal.** Report **pair accuracy** (the share of pairs where the checker scores the supporting evidence above the refuting evidence; it needs no threshold) and accuracy at a 0.5 threshold, each with a Wilson interval. The Wilson interval stays inside 0 to 1 and widens correctly for few items, which the usual normal interval does not.

In [5]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def summarize(R, title):
    by = collections.defaultdict(dict)
    for r in R: by[r["model"]].setdefault(r["pair"], {})[r["side"]] = r
    print(title)
    for m, d in by.items():
        n = len(d); pa = sum(v["S"]["score"] > v["R"]["score"] for v in d.values()); lo, hi = wilson(pa, n)
        line = f"  {m:10s} pair accuracy {pa}/{n} = {pa/n:.3f}  [{lo:.3f}, {hi:.3f}]"
        if m != "ce":
            ex = [x for v in d.values() for x in v.values()]; a = sum((x["score"] > 0.5) == bool(x["y"]) for x in ex); lo, hi = wilson(a, len(ex))
            line += f" | at 0.5: {a}/{len(ex)} = {a/len(ex):.3f}  [{lo:.3f}, {hi:.3f}]"
        print(line)
summarize(fresh, f"This run, {N_PAIRS} pairs")
recorded = [json.loads(l) for l in open(RECORD / "scores_300.jsonl")]
summarize(recorded, "Recorded run, 300 pairs (3 November 2026, sorted-claim pair selection)")

This run, 50 pairs
  ce         pair accuracy 30/50 = 0.600  [0.462, 0.724]
  nli        pair accuracy 43/50 = 0.860  [0.738, 0.930] | at 0.5: 65/100 = 0.650  [0.553, 0.736]
  hhem       pair accuracy 32/50 = 0.640  [0.501, 0.759] | at 0.5: 57/100 = 0.570  [0.472, 0.663]
  minicheck  pair accuracy 48/50 = 0.960  [0.865, 0.989] | at 0.5: 92/100 = 0.920  [0.850, 0.959]
Recorded run, 300 pairs (3 November 2026, sorted-claim pair selection)
  minicheck  pair accuracy 294/300 = 0.980  [0.957, 0.991] | at 0.5: 567/600 = 0.945  [0.924, 0.961]
  hhem       pair accuracy 217/300 = 0.723  [0.670, 0.771] | at 0.5: 386/600 = 0.643  [0.604, 0.681]
  nli        pair accuracy 238/300 = 0.793  [0.744, 0.835] | at 0.5: 409/600 = 0.682  [0.643, 0.718]
  ce         pair accuracy 184/300 = 0.613  [0.557, 0.667]


**Goal.** Repeat the test on Meridian text. Three pairs come from the Meridian policy corpus (the 2019 and 2025 baggage fees, the 2021 and 2025 entry rules, the 2023 and 2025 Schengen guidance), and three are written for this notebook. In each pair the claim is true under the **current** document. A date rule (take the document with the later `effective_date`) is the baseline that needs no model. It gets every pair right here because the dates are in the metadata, which is the point: where the date is known, use the rule; the checker is for evidence whose date is not.

In [6]:
def doc(name):
    head, body = (Path("data/rag/meridian/policies") / f"{name}.md").read_text().split("---\n", 2)[1:]
    meta = dict(l.split(": ", 1) for l in head.strip().splitlines() if ": " in l)
    return meta["effective_date"], " ".join(body.split())

meridian = [  # (claim true under the current document, stale document text or name, current document text or name)
    ("A 23 kg checked bag costs 40 GBP each way when added in advance.", "baggage_2019_outdated", "baggage_current"),
    ("No COVID-19 vaccination or test is required for entry to any Meridian destination.", "health_covid_archived", "health_current"),
    ("From 2025 the EU is phasing in ETIAS, a paid online travel authorisation for visa-exempt visitors.", "visa_schengen_2023_archived", "visa_schengen"),
    # written for this notebook, dated by hand
    ("Premium fares are fully refundable up to 14 days before departure.",
     ("2022-01-01", "Premium fares are fully refundable up to 30 days before departure."),
     ("2025-01-01", "Premium fares are fully refundable up to 14 days before departure.")),
    ("Meridian charges 55 GBP for a checked bag added at the airport.",
     ("2022-01-01", "A checked bag added at the airport costs 45 GBP."),
     ("2025-01-01", "A checked bag added at the airport costs 55 GBP.")),
    ("Children under 2 travel on a package fare for free.",
     ("2021-01-01", "Children under 2 travel on a package fare at 10 percent of the adult price."),
     ("2025-01-01", "Children under 2 travel on a package fare for free.")),
]
resolve = lambda x: doc(x) if isinstance(x, str) else x
print("claim | " + " | ".join(scorers))
mer = []
for claim, stale, cur in meridian:
    (d0, t0), (d1, t1) = resolve(stale), resolve(cur)
    row = {n: (f(claim, t1), f(claim, t0)) for n, f in scorers.items()}
    mer.append(row)
    print(claim[:48].ljust(48), "| " + " | ".join(f"{n}: current {a:.2f} stale {b:.2f}" for n, (a, b) in row.items()))
for n in scorers:
    k = sum(r[n][0] > r[n][1] for r in mer); lo, hi = wilson(k, len(mer))
    print(f"{n:10s} ranks the current document higher in {k} of {len(mer)} pairs  [{lo:.2f}, {hi:.2f}]")
print("date rule  picks the later effective_date in", sum(resolve(c)[0] > resolve(s)[0] for _, s, c in meridian), "of", len(meridian), "pairs")

claim | ce | nli | hhem | minicheck


A 23 kg checked bag costs 40 GBP each way when a | ce: current 8.72 stale 8.23 | nli: current 0.00 stale 0.00 | hhem: current 0.95 stale 0.02 | minicheck: current 0.98 stale 0.01


No COVID-19 vaccination or test is required for  | ce: current 9.28 stale 3.68 | nli: current 0.07 stale 0.00 | hhem: current 0.98 stale 0.09 | minicheck: current 0.93 stale 0.01


From 2025 the EU is phasing in ETIAS, a paid onl | ce: current 7.44 stale 1.90 | nli: current 0.00 stale 0.00 | hhem: current 0.99 stale 0.06 | minicheck: current 0.99 stale 0.01


Premium fares are fully refundable up to 14 days | ce: current 10.16 stale 8.97 | nli: current 0.99 stale 0.00 | hhem: current 0.92 stale 0.05 | minicheck: current 0.99 stale 0.04


Meridian charges 55 GBP for a checked bag added  | ce: current 5.62 stale 4.36 | nli: current 0.00 stale 0.00 | hhem: current 0.19 stale 0.02 | minicheck: current 0.08 stale 0.04


Children under 2 travel on a package fare for fr | ce: current 9.96 stale 7.06 | nli: current 0.99 stale 0.00 | hhem: current 0.92 stale 0.00 | minicheck: current 0.99 stale 0.08
ce         ranks the current document higher in 6 of 6 pairs  [0.61, 1.00]
nli        ranks the current document higher in 6 of 6 pairs  [0.61, 1.00]
hhem       ranks the current document higher in 6 of 6 pairs  [0.61, 1.00]
minicheck  ranks the current document higher in 6 of 6 pairs  [0.61, 1.00]
date rule  picks the later effective_date in 6 of 6 pairs


**Goal.** Six pairs cannot separate the checkers, so repeat the test on 118 Meridian superseded-policy pairs. They were written by an LLM (Claude, prompted for six kinds of change: a number, a threshold, a reversed rule, an added or removed condition, a scope and a deadline; the other 2 of 120 were dropped as unclear), not by a person, and the claim in each pair paraphrases the current text. That makes them easier than the Wikipedia pairs, where the claim is the same in both versions and only the edit differs; read the result as a floor on how much a relevance score can do when the claim is written from the current policy. The pairs and their scores are recorded in `data/results/17_grounding_checkers/`.

In [7]:
import collections
from scipy.stats import binomtest
mp = {r["id"]: r for r in map(json.loads, open(RECORD / "meridian_pairs_llm_written.jsonl"))}
ms = [json.loads(l) for l in open(RECORD / "meridian_scores.jsonl")]
mby = collections.defaultdict(dict)
for r in ms: mby[r["model"]].setdefault(r["pair"], {})[r["side"]] = r
right = {mdl: {p: v["S"]["score"] > v["R"]["score"] for p, v in d.items()} for mdl, d in mby.items()}
for mdl, d in right.items():
    k, n = sum(d.values()), len(d); lo, hi = wilson(k, n)
    print(f"{mdl:10s} ranks the current text higher in {k} of {n} pairs ({k/n:.3f}, Wilson {lo:.3f} to {hi:.3f})")
a, b = right["minicheck"], right["ce"]
n10 = sum(a[p] and not b[p] for p in a); n01 = sum(b[p] and not a[p] for p in a)
print(f"MiniCheck right and cross-encoder wrong on {n10} pairs, the reverse on {n01}; exact McNemar p = {binomtest(n10, n10 + n01, 0.5).pvalue:.3f}")
bytype = collections.defaultdict(lambda: collections.Counter())
for p, v in mby["ce"].items():
    t = mp[v["S"]["case"]]["change_type"]; bytype[t]["n"] += 1; bytype[t]["ce"] += right["ce"][p]; bytype[t]["mc"] += right["minicheck"][p]
for t, c in sorted(bytype.items()): print(f"  {t:28s} cross-encoder {c['ce']:2d}/{c['n']}  MiniCheck {c['mc']:2d}/{c['n']}")

ce         ranks the current text higher in 107 of 118 pairs (0.907, Wilson 0.841 to 0.947)
nli        ranks the current text higher in 110 of 118 pairs (0.932, Wilson 0.872 to 0.965)
hhem       ranks the current text higher in 109 of 118 pairs (0.924, Wilson 0.861 to 0.959)
minicheck  ranks the current text higher in 111 of 118 pairs (0.941, Wilson 0.883 to 0.971)
MiniCheck right and cross-encoder wrong on 6 pairs, the reverse on 2; exact McNemar p = 0.289
  condition_added_or_removed   cross-encoder 14/18  MiniCheck 15/18
  deadline_changed             cross-encoder 19/20  MiniCheck 19/20
  number_changed               cross-encoder 20/20  MiniCheck 20/20
  rule_reversed                cross-encoder 16/20  MiniCheck 18/20
  scope_changed                cross-encoder 20/20  MiniCheck 19/20
  threshold_moved              cross-encoder 18/20  MiniCheck 20/20


**Goal.** Add a small prompted language model as a fifth checker: Qwen3-1.7B (a 1.7B-parameter chat model, thinking switched off) gets the evidence and the claim and is asked "Is the claim supported by the evidence? Answer with one word, Yes or No." The score is the probability of "Yes" against "No" from one forward pass, so it costs the same shape of call as the others. It was scored on the same 300 pairs in a separate run (`llm_judge.py`; its timings were taken while the machine was loaded and are not reported).

In [8]:
judge = [json.loads(l) for l in open(RECORD / "qwen3_1.7b_judge_300.jsonl")]
jb = collections.defaultdict(dict)
for r in judge: jb[r["pair"]][r["side"]] = r
jright = {p: v["S"]["score"] > v["R"]["score"] for p, v in jb.items()}
k, n = sum(jright.values()), len(jright); lo, hi = wilson(k, n)
print(f"Qwen3-1.7B prompted judge ranks the supporting evidence higher in {k} of {n} pairs ({k/n:.3f}, Wilson {lo:.3f} to {hi:.3f}); accuracy at 0.5: {np.mean([(r['score'] > 0.5) == bool(r['y']) for r in judge]):.3f}")
rec_by = collections.defaultdict(dict)
for r in recorded: rec_by[r["model"]].setdefault(r["pair"], {})[r["side"]] = r
for mdl in ("minicheck", "nli", "hhem", "ce"):
    cr = {p: rec_by[mdl][p]["S"]["score"] > rec_by[mdl][p]["R"]["score"] for p in jright}
    n10 = sum(jright[p] and not cr[p] for p in jright); n01 = sum(cr[p] and not jright[p] for p in jright)
    print(f"  against {mdl:9s}: judge only right {n10:3d}, {mdl} only right {n01:3d}, exact McNemar p = {binomtest(n10, n10 + n01, 0.5).pvalue:.2g}")

Qwen3-1.7B prompted judge ranks the supporting evidence higher in 255 of 300 pairs (0.850, Wilson 0.805 to 0.886); accuracy at 0.5: 0.700
  against minicheck: judge only right   3, minicheck only right  42, exact McNemar p = 8.7e-10
  against nli      : judge only right  45, nli only right  28, exact McNemar p = 0.06
  against hhem     : judge only right  59, hhem only right  21, exact McNemar p = 2.5e-05
  against ce       : judge only right  94, ce only right  23, exact McNemar p = 2.2e-11


**Goal.** Run the same prompted Qwen3-1.7B judge on the 118 Meridian pairs, and compare it with each trained checker on the same pairs with an exact McNemar test.

In [9]:
jm = collections.defaultdict(dict)
for r in map(json.loads, open(RECORD / "qwen3_1.7b_judge_meridian118.jsonl")): jm[r["pair"]][r["side"]] = r["score"]
jr = {p: v["S"] > v["R"] for p, v in jm.items()}
k, n = sum(jr.values()), len(jr); lo, hi = wilson(k, n)
print(f"prompted Qwen3-1.7B ranks the current text higher in {k} of {n} Meridian pairs ({k/n:.3f}, Wilson {lo:.3f} to {hi:.3f})")
for mdl in ("minicheck", "nli", "hhem", "ce"):
    d = right[mdl]
    n10 = sum(jr[p] and not d[p] for p in jr); n01 = sum(d[p] and not jr[p] for p in jr)
    print(f"  against {mdl:9s}: judge only right {n10:3d}, {mdl} only right {n01:3d}, exact McNemar p = {binomtest(n10, n10 + n01, 0.5).pvalue:.2g}")

prompted Qwen3-1.7B ranks the current text higher in 112 of 118 Meridian pairs (0.949, Wilson 0.893 to 0.976)
  against minicheck: judge only right   6, minicheck only right   5, exact McNemar p = 1
  against nli      : judge only right   7, nli only right   5, exact McNemar p = 0.77
  against hhem     : judge only right   9, hhem only right   6, exact McNemar p = 0.61
  against ce       : judge only right  10, ce only right   5, exact McNemar p = 0.3


**Goal.** From the recorded 300 pairs, fit a **split conformal** abstention gate. Half the pairs (150, 300 examples) calibrate it and the other half test it. For each example the checker gives a probability *p* that the evidence supports the claim, and the nonconformity score is one minus the probability of the true answer. The gate takes the 95% conformal quantile *q* of those scores on the calibration half; on a new example it keeps every answer (supported, refuted) whose one-minus-probability is at most *q*. If one answer remains, the checker decides alone. If both or neither remain, a person or the slow tier decides. Under exchangeability (calibration and test drawn from one pool) the true answer lands in the kept set about 95% of the time, on average over splits. The result below averages 200 random splits made by pair, so the two examples of one claim never straddle the split. **Expected calibration error (ECE)** compares the stated probability with the observed accuracy in ten equal-width bins; it changes with the number of bins.

In [10]:
by = collections.defaultdict(dict)
for r in recorded: by[r["model"]].setdefault(r["pair"], {})[r["side"]] = r

def ece(ps, ys, bins=10):
    tot = 0
    for b in range(bins):
        idx = [i for i, p in enumerate(ps) if p >= b / bins and (p < (b + 1) / bins or b == bins - 1)]
        if idx: tot += len(idx) * abs(np.mean([ps[i] for i in idx]) - np.mean([ys[i] for i in idx]))
    return tot / len(ps)

def conformal(m, alpha, seed):
    d = by[m]; ids = sorted(d); random.Random(seed).shuffle(ids); cal, te = ids[:150], ids[150:]
    ex = lambda S: [(d[i][s]["score"], d[i][s]["y"]) for i in S for s in "SR"]
    c, t = ex(cal), ex(te)
    ns = sorted(1 - (p if y else 1 - p) for p, y in c)
    q = ns[min(math.ceil((len(ns) + 1) * (1 - alpha)) - 1, len(ns) - 1)]
    cov = sing = ok = 0
    for p, y in t:
        S = [lab for lab, pr in ((1, p), (0, 1 - p)) if 1 - pr <= q]
        cov += y in S
        if len(S) == 1: sing += 1; ok += S[0] == y
    return cov / len(t), sing / len(t), ok / sing if sing else float("nan"), ece([p for p, _ in t], [y for _, y in t])

conf_out = {}
for alpha in (0.10, 0.05):
    print(f"target coverage {1 - alpha:.0%}")
    for m in ("minicheck", "hhem", "nli"):
        res = np.array([conformal(m, alpha, s) for s in range(200)])
        conf_out[f"{m}_alpha_{alpha}"] = dict(coverage=float(res[:,0].mean()), worst_coverage=float(res[:,0].min()), decided_alone=float(res[:,1].mean()),
                                              right_when_decided=float(np.nanmean(res[:,2])), ece=float(res[:,3].mean()))
        print(f"  {m:10s} coverage {res[:,0].mean():.3f} (worst split {res[:,0].min():.3f}) | decided alone {res[:,1].mean():.3f} | right when decided alone {np.nanmean(res[:,2]):.3f} | ECE {res[:,3].mean():.3f}")

target coverage 90%


  minicheck  coverage 0.900 (worst split 0.830) | decided alone 0.927 | right when decided alone 0.971 | ECE 0.065


  hhem       coverage 0.900 (worst split 0.840) | decided alone 0.464 | right when decided alone 0.788 | ECE 0.141


  nli        coverage 0.901 (worst split 0.820) | decided alone 0.343 | right when decided alone 0.714 | ECE 0.304
target coverage 95%


  minicheck  coverage 0.951 (worst split 0.873) | decided alone 0.980 | right when decided alone 0.954 | ECE 0.065


  hhem       coverage 0.951 (worst split 0.877) | decided alone 0.322 | right when decided alone 0.850 | ECE 0.141


  nli        coverage 0.951 (worst split 0.903) | decided alone 0.216 | right when decided alone 0.778 | ECE 0.304


**Goal.** Write the numbers the book and the slides quote to one file, so they cannot drift from the code.

In [11]:
pair_acc = {}
byr = collections.defaultdict(dict)
for r in recorded: byr[r["model"]].setdefault(r["pair"], {})[r["side"]] = r
for m, d in byr.items():
    k = sum(v["S"]["score"] > v["R"]["score"] for v in d.values()); n = len(d); lo, hi = wilson(k, n)
    ex = [x for v in d.values() for x in v.values()]
    a = sum((x["score"] > 0.5) == bool(x["y"]) for x in ex) if m != "ce" else None
    pair_acc[m] = dict(k=k, n=n, acc=k / n, lo=lo, hi=hi, acc_at_half=(a / len(ex) if a is not None else None))
mer_acc = {n: sum(r[n][0] > r[n][1] for r in mer) for n in scorers}
out = dict(pairs=pair_acc, conformal=conf_out, meridian_pairs=len(mer), meridian_correct=mer_acc,
           note="VitaminC test split, 300 flip pairs, seed 0; conformal averaged over 200 pair-level 150/150 splits")
Path("data/results/17_grounding_checkers.json").write_text(json.dumps(out, indent=1))
print(json.dumps(out["pairs"], indent=0)[:400])

{
"minicheck": {
"k": 294,
"n": 300,
"acc": 0.98,
"lo": 0.9570598132699448,
"hi": 0.9908024939322289,
"acc_at_half": 0.945
},
"hhem": {
"k": 217,
"n": 300,
"acc": 0.7233333333333334,
"lo": 0.6701290366526463,
"hi": 0.7708902312816984,
"acc_at_half": 0.6433333333333333
},
"nli": {
"k": 238,
"n": 300,
"acc": 0.7933333333333333,
"lo": 0.7439440144288735,
"hi": 0.8353051733057882,
"acc_at_half": 0.681


In [12]:
P, C = out["pairs"], out["conformal"]
f = lambda m: f"{P[m]['k']} of {P[m]['n']} pairs ({P[m]['acc']:.3f}, Wilson {P[m]['lo']:.3f} to {P[m]['hi']:.3f})"
c95 = lambda m: C[f"{m}_alpha_0.05"]
print("Relevance score (cross-encoder):", f("ce"))
print("Entailment checkers: MiniCheck", f("minicheck"), "| NLI", f("nli"), "| HHEM", f("hhem"))
print("At a 0.5 threshold, accuracy is MiniCheck %.3f, NLI %.3f, HHEM %.3f" % tuple(P[m]["acc_at_half"] for m in ("minicheck", "nli", "hhem")))
for m in ("minicheck", "nli", "hhem"):
    c = c95(m); print(f"Conformal gate at 95% target, {m}: coverage {c['coverage']:.3f} (worst split {c['worst_coverage']:.3f}), decides alone {c['decided_alone']:.3f}, right when it does {c['right_when_decided']:.3f}")
print("Meridian pairs ranked right:", out["meridian_correct"], "of", out["meridian_pairs"])

Relevance score (cross-encoder): 184 of 300 pairs (0.613, Wilson 0.557 to 0.667)
Entailment checkers: MiniCheck 294 of 300 pairs (0.980, Wilson 0.957 to 0.991) | NLI 238 of 300 pairs (0.793, Wilson 0.744 to 0.835) | HHEM 217 of 300 pairs (0.723, Wilson 0.670 to 0.771)
At a 0.5 threshold, accuracy is MiniCheck 0.945, NLI 0.682, HHEM 0.643
Conformal gate at 95% target, minicheck: coverage 0.951 (worst split 0.873), decides alone 0.980, right when it does 0.954
Conformal gate at 95% target, nli: coverage 0.951 (worst split 0.903), decides alone 0.216, right when it does 0.778
Conformal gate at 95% target, hhem: coverage 0.951 (worst split 0.877), decides alone 0.322, right when it does 0.850
Meridian pairs ranked right: {'ce': 6, 'nli': 6, 'hhem': 6, 'minicheck': 6} of 6


**What to take from this.**

- **Relevance is the wrong score for "is this still true".** On the VitaminC pairs the cross-encoder separates stale from current evidence only a little better than a coin flip (the first printed line above). On the six hand-picked Meridian pairs every checker ranks the current text higher. On 118 LLM-written Meridian pairs the cross-encoder is right on about 0.91 and MiniCheck on about 0.94, intervals that overlap (printed above), because each claim paraphrases the current text and a relevance score rewards that overlap. The relevance-score failure needs claims whose wording fits both versions, as in the Wikipedia pairs.
- **An entailment-trained checker works, and size alone does not decide it.** MiniCheck-RoBERTa-Large (355M) is far ahead of the NLI model (184M) and HHEM, and moving their threshold on a calibration split did not rescue their accuracy at 0.5, so their weakness is ranking, not the cut-off.
- **A conformal gate turns the good checker into a decision rule.** At a 95% target MiniCheck decides most examples alone and is right on about that share; the others send most of their traffic to the slow tier.
- **A ranking is not a calibrated probability.** MiniCheck ranked the written airport-fee pair correctly but scored the true current claim 0.08, so a 0.5 threshold would call it unsupported. Calibrate on your own text before using a number as a decision.

Limits. VitaminC is Wikipedia text with one pair per case, the calibration and test halves come from the same pool, and the coverage guarantee is about averages over splits (the worst of 200 splits is printed above). It says nothing about drift, and the six Meridian pairs are an illustration whose intervals are wide. The pair set is fixed by a seed and sorted claims; an earlier run that let Python's set order choose the claim produced different pairs from the same cases, which is why the recorded scores in `data/results/17_grounding_checkers/` are the second recorded run. The timings of the recorded run were taken under varying machine load and are not reported.

**Threshold for your own system.** Score at least 100 stale-versus-current pairs from your own documents. If the lower Wilson bound on pair accuracy is below 0.90, send the pair to a date rule or a person instead of letting the checker decide.